# Code Generator

The requirement: use an Open Source model to generate high performance C++ code from Python code

To replicate this, you'll need to set up a HuggingFace endpoint as I do in the video. It's simple to do, and it's quite satisfying to see the results!

It's also an important part of your learning; this is the first example of deploying an open source model to be behind an API. We'll return to this in Week 8, but this should plant a seed in your mind for what's involved in moving open source models into production.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important - Pause Endpoints when not in use</h1>
            <span style="color:#900;">
            If you do decide to use HuggingFace endpoints for this project, you should stop or pause the endpoints when you are done to avoid accruing unnecessary running cost. The costs are very low as long as you only run the endpoint when you're using it. Navigate to the HuggingFace endpoint UI <a href="https://ui.endpoints.huggingface.co/">here,</a> open your endpoint, and click Pause to put it on pause so you no longer pay for it.  
Many thanks to student John L. for raising this.
<br/><br/>
In week 8 we will use Modal instead of HuggingFace endpoints; with Modal you only pay for the time that you use it and you should get free credits.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
import io
import sys
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai
import anthropic
from IPython.display import Markdown, display, update_display
import gradio as gr
import subprocess

In [2]:
# environment

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')

In [3]:
# initialize

openai = OpenAI()
claude = anthropic.Anthropic()

# Configure Google Gemini
google.generativeai.configure(api_key=os.environ['GOOGLE_API_KEY'])
gemini = google.generativeai.GenerativeModel('gemini-1.5-flash')

OPENAI_MODEL = "gpt-4o"
CLAUDE_MODEL = "claude-3-5-sonnet-20240620"
GEMINI_MODEL = "gemini-1.5-flash"

In [4]:
system_message = "You are an assistant that reimplements Python code in high performance C++ for an M1 Mac. "
system_message += "Respond only with C++ code; use comments sparingly and do not provide any explanation other than occasional comments. "
system_message += "The C++ response needs to produce an identical output in the fastest possible time. Keep implementations of random number generators identical so that results match exactly."

In [5]:
def user_prompt_for(python):
    user_prompt = "Rewrite this Python code in C++ with the fastest possible implementation that produces identical output in the least time. "
    user_prompt += "Respond only with C++ code; do not explain your work other than a few comments. "
    user_prompt += "Pay attention to number types to ensure no int overflows. Remember to #include all necessary C++ packages such as iomanip.\n\n"
    user_prompt += python
    return user_prompt

In [6]:
def messages_for(python):
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [7]:
# write to a file called optimized.cpp

def write_output(cpp):
    code = cpp.replace("```cpp","").replace("```","")
    with open("optimized.cpp", "w") as f:
        f.write(code)

In [8]:
def optimize_gpt(python):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(python), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        print(fragment, end='', flush=True)
    write_output(reply)

In [9]:
def optimize_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            print(text, end="", flush=True)
    write_output(reply)

In [10]:
def optimize_gemini(python):
    # Create the prompt by combining system message and user prompt
    full_prompt = system_message + "\n\n" + user_prompt_for(python)
    
    try:
        response = gemini.generate_content(
            full_prompt,
            stream=True,
            generation_config=google.generativeai.types.GenerationConfig(
                max_output_tokens=2000,
                temperature=0.1
            )
        )
        
        reply = ""
        for chunk in response:
            if chunk.text:
                reply += chunk.text
                print(chunk.text, end="", flush=True)
        
        write_output(reply)
        
    except Exception as e:
        error_msg = f"Error with Gemini: {str(e)}"
        print(error_msg)
        write_output(error_msg)

In [11]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(100_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [12]:
exec(pi)

Result: 3.141592658589
Execution Time: 7.550641 seconds


In [13]:
optimize_gpt(pi)

```cpp
#include <iostream>
#include <iomanip>
#include <chrono>

double calculate(int iterations, int param1, int param2) {
    double result = 1.0;
    for (int i = 1; i <= iterations; ++i) {
        int j = i * param1 - param2;
        result -= (1.0 / j);
        j = i * param1 + param2;
        result += (1.0 / j);
    }
    return result;
}

int main() {
    using namespace std::chrono;

    auto start_time = high_resolution_clock::now();
    double result = calculate(100'000'000, 4, 1) * 4;
    auto end_time = high_resolution_clock::now();

    duration<double> exec_time = end_time - start_time;

    std::cout << std::fixed << std::setprecision(12);
    std::cout << "Result: " << result << '\n';
    std::cout << "Execution Time: " << exec_time.count() << " seconds" << std::endl;

    return 0;
}
```

In [14]:
exec(pi)

Result: 3.141592658589
Execution Time: 7.677047 seconds


In [15]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

error: unknown target CPU 'armv8.3-a'
note: valid target CPU values are: nocona, core2, penryn, bonnell, atom, silvermont, slm, goldmont, goldmont-plus, tremont, nehalem, corei7, westmere, sandybridge, corei7-avx, ivybridge, core-avx-i, haswell, core-avx2, broadwell, skylake, skylake-avx512, skx, cascadelake, cooperlake, cannonlake, icelake-client, rocketlake, icelake-server, tigerlake, sapphirerapids, alderlake, raptorlake, meteorlake, arrowlake, arrowlake-s, lunarlake, gracemont, pantherlake, sierraforest, grandridge, graniterapids, graniterapids-d, emeraldrapids, clearwaterforest, knl, knm, k8, athlon64, athlon-fx, opteron, k8-sse3, athlon64-sse3, opteron-sse3, amdfam10, barcelona, btver1, btver2, bdver1, bdver2, bdver3, bdver4, znver1, znver2, znver3, znver4, x86-64, x86-64-v2, x86-64-v3, x86-64-v4


In [16]:
optimize_claude(pi)

/tmp/ipykernel_24837/4229984635.py:1: DeprecationWarning: The model 'claude-3-5-sonnet-20240620' is deprecated and will reach end-of-life on October 22, 2025.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  optimize_claude(pi)


#include <iostream>
#include <iomanip>
#include <chrono>

double calculate(long long iterations, int param1, int param2) {
    double result = 1.0;
    #pragma omp parallel for reduction(-:result)
    for (long long i = 1; i <= iterations; ++i) {
        double j = i * static_cast<double>(param1) - param2;
        result -= (1.0 / j);
        j = i * static_cast<double>(param1) + param2;
        result += (1.0 / j);
    }
    return result;
}

int main() {
    auto start_time = std::chrono::high_resolution_clock::now();
    
    double result = calculate(100'000'000, 4, 1) * 4;
    
    auto end_time = std::chrono::high_resolution_clock::now();
    auto duration = std::chrono::duration_cast<std::chrono::microseconds>(end_time - start_time);

    std::cout << "Result: " << std::fixed << std::setprecision(12) << result << std::endl;
    std::cout << "Execution Time: " << duration.count() / 1e6 << " seconds" << std::endl;

    return 0;
}

In [17]:
optimize_gemini(pi)

```cpp
#include <iostream>
#include <iomanip>
#include <chrono>

using namespace std;
using namespace chrono;

double calculate(long long iterations, long long param1, long long param2) {
    double result = 1.0;
    for (long long i = 1; i <= iterations; ++i) {
        long double j = (long double)i * param1 - param2;
        result -= (1.0L / j);
        j = (long double)i * param1 + param2;
        result += (1.0L / j);
    }
    return result;
}

int main() {
    auto start_time = high_resolution_clock::now();
    double result = calculate(100000000, 4, 1) * 4;
    auto end_time = high_resolution_clock::now();

    auto duration = duration_cast<microseconds>(end_time - start_time);

    cout << "Result: " << fixed << setprecision(12) << result << endl;
    cout << "Execution Time: " << (double)duration.count() / 1000000.0 << " seconds" << endl;

    return 0;
}
```


In [18]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

error: unknown target CPU 'armv8.3-a'
note: valid target CPU values are: nocona, core2, penryn, bonnell, atom, silvermont, slm, goldmont, goldmont-plus, tremont, nehalem, corei7, westmere, sandybridge, corei7-avx, ivybridge, core-avx-i, haswell, core-avx2, broadwell, skylake, skylake-avx512, skx, cascadelake, cooperlake, cannonlake, icelake-client, rocketlake, icelake-server, tigerlake, sapphirerapids, alderlake, raptorlake, meteorlake, arrowlake, arrowlake-s, lunarlake, gracemont, pantherlake, sierraforest, grandridge, graniterapids, graniterapids-d, emeraldrapids, clearwaterforest, knl, knm, k8, athlon64, athlon-fx, opteron, k8-sse3, athlon64-sse3, opteron-sse3, amdfam10, barcelona, btver1, btver2, bdver1, bdver2, bdver3, bdver4, znver1, znver2, znver3, znver4, x86-64, x86-64-v2, x86-64-v3, x86-64-v4


In [19]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

error: unknown target CPU 'armv8.3-a'
note: valid target CPU values are: nocona, core2, penryn, bonnell, atom, silvermont, slm, goldmont, goldmont-plus, tremont, nehalem, corei7, westmere, sandybridge, corei7-avx, ivybridge, core-avx-i, haswell, core-avx2, broadwell, skylake, skylake-avx512, skx, cascadelake, cooperlake, cannonlake, icelake-client, rocketlake, icelake-server, tigerlake, sapphirerapids, alderlake, raptorlake, meteorlake, arrowlake, arrowlake-s, lunarlake, gracemont, pantherlake, sierraforest, grandridge, graniterapids, graniterapids-d, emeraldrapids, clearwaterforest, knl, knm, k8, athlon64, athlon-fx, opteron, k8-sse3, athlon64-sse3, opteron-sse3, amdfam10, barcelona, btver1, btver2, bdver1, bdver2, bdver3, bdver4, znver1, znver2, znver3, znver4, x86-64, x86-64-v2, x86-64-v3, x86-64-v4


In [20]:
python_hard = """# Be careful to support large number sizes

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [21]:
exec(python_hard)

Total Maximum Subarray Sum (20 runs): 10980
Execution Time: 25.855053 seconds


In [22]:
optimize_gpt(python_hard)

```cpp
#include <iostream>
#include <vector>
#include <chrono>
#include <limits>
#include <cstdint>

// Parameters
const int n = 10000;
const uint32_t initial_seed = 42;
const int min_val = -10;
const int max_val = 10;

class LCG {
public:
    LCG(uint32_t seed, uint32_t a = 1664525, uint32_t c = 1013904223, uint32_t m = (1U << 32))
        : value(seed), a(a), c(c), m(m) {}
    
    uint32_t next() {
        value = (a * value + c) % m;
        return value;
    }

private:
    uint32_t value;
    uint32_t a;
    uint32_t c;
    uint32_t m;
};

// Find the maximum subarray sum in a vector of integers
int max_subarray_sum(const std::vector<int>& numbers) {
    int max_sum = std::numeric_limits<int>::min();
    int current_sum = 0;
    
    for (int number : numbers) {
        current_sum += number;
        if (current_sum > max_sum) {
            max_sum = current_sum;
        }
        if (current_sum < 0) {
            current_sum = 0;
        }
    }

    return max_sum;
}

// Gener

In [23]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

error: unknown target CPU 'armv8.3-a'
note: valid target CPU values are: nocona, core2, penryn, bonnell, atom, silvermont, slm, goldmont, goldmont-plus, tremont, nehalem, corei7, westmere, sandybridge, corei7-avx, ivybridge, core-avx-i, haswell, core-avx2, broadwell, skylake, skylake-avx512, skx, cascadelake, cooperlake, cannonlake, icelake-client, rocketlake, icelake-server, tigerlake, sapphirerapids, alderlake, raptorlake, meteorlake, arrowlake, arrowlake-s, lunarlake, gracemont, pantherlake, sierraforest, grandridge, graniterapids, graniterapids-d, emeraldrapids, clearwaterforest, knl, knm, k8, athlon64, athlon-fx, opteron, k8-sse3, athlon64-sse3, opteron-sse3, amdfam10, barcelona, btver1, btver2, bdver1, bdver2, bdver3, bdver4, znver1, znver2, znver3, znver4, x86-64, x86-64-v2, x86-64-v3, x86-64-v4


In [24]:
optimize_claude(python_hard)

/tmp/ipykernel_24837/92554486.py:1: DeprecationWarning: The model 'claude-3-5-sonnet-20240620' is deprecated and will reach end-of-life on October 22, 2025.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  optimize_claude(python_hard)


#include <iostream>
#include <vector>
#include <chrono>
#include <limits>
#include <iomanip>

using namespace std;
using namespace chrono;

class LCG {
private:
    uint64_t value;
    const uint64_t a = 1664525;
    const uint64_t c = 1013904223;
    const uint64_t m = 1ULL << 32;

public:
    LCG(uint64_t seed) : value(seed) {}

    uint64_t next() {
        value = (a * value + c) % m;
        return value;
    }
};

int64_t max_subarray_sum(int n, uint64_t seed, int min_val, int max_val) {
    LCG lcg_gen(seed);
    vector<int64_t> random_numbers(n);
    for (int i = 0; i < n; ++i) {
        random_numbers[i] = lcg_gen.next() % (max_val - min_val + 1) + min_val;
    }

    int64_t max_sum = numeric_limits<int64_t>::min();
    int64_t current_sum = 0;
    int64_t min_sum = 0;

    for (int i = 0; i < n; ++i) {
        current_sum += random_numbers[i];
        max_sum = max(max_sum, current_sum - min_sum);
        min_sum = min(min_sum, current_sum);
    }

    return max_sum;
}

int

In [25]:
optimize_gemini(python_hard)

```cpp
#include <iostream>
#include <vector>
#include <limits> // Required for numeric_limits
#include <chrono> // Required for high_resolution_clock

using namespace std;

// Linear Congruential Generator
unsigned long long lcg(unsigned long long seed, unsigned long long a = 1664525, unsigned long long c = 1013904223, unsigned long long m = 4294967296ULL) {
    return (a * seed + c) % m;
}

long long max_subarray_sum(int n, unsigned long long seed, long long min_val, long long max_val) {
    vector<long long> random_numbers(n);
    unsigned long long current_seed = seed;
    for (int i = 0; i < n; ++i) {
        current_seed = lcg(current_seed);
        random_numbers[i] = (current_seed % (max_val - min_val + 1)) + min_val;
    }

    long long max_sum = numeric_limits<long long>::min();
    for (int i = 0; i < n; ++i) {
        long long current_sum = 0;
        for (int j = i; j < n; ++j) {
            current_sum += random_numbers[j];
            if (current_sum > max_sum) {
      

In [26]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

error: unknown target CPU 'armv8.3-a'
note: valid target CPU values are: nocona, core2, penryn, bonnell, atom, silvermont, slm, goldmont, goldmont-plus, tremont, nehalem, corei7, westmere, sandybridge, corei7-avx, ivybridge, core-avx-i, haswell, core-avx2, broadwell, skylake, skylake-avx512, skx, cascadelake, cooperlake, cannonlake, icelake-client, rocketlake, icelake-server, tigerlake, sapphirerapids, alderlake, raptorlake, meteorlake, arrowlake, arrowlake-s, lunarlake, gracemont, pantherlake, sierraforest, grandridge, graniterapids, graniterapids-d, emeraldrapids, clearwaterforest, knl, knm, k8, athlon64, athlon-fx, opteron, k8-sse3, athlon64-sse3, opteron-sse3, amdfam10, barcelona, btver1, btver2, bdver1, bdver2, bdver3, bdver4, znver1, znver2, znver3, znver4, x86-64, x86-64-v2, x86-64-v3, x86-64-v4


In [27]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

error: unknown target CPU 'armv8.3-a'
note: valid target CPU values are: nocona, core2, penryn, bonnell, atom, silvermont, slm, goldmont, goldmont-plus, tremont, nehalem, corei7, westmere, sandybridge, corei7-avx, ivybridge, core-avx-i, haswell, core-avx2, broadwell, skylake, skylake-avx512, skx, cascadelake, cooperlake, cannonlake, icelake-client, rocketlake, icelake-server, tigerlake, sapphirerapids, alderlake, raptorlake, meteorlake, arrowlake, arrowlake-s, lunarlake, gracemont, pantherlake, sierraforest, grandridge, graniterapids, graniterapids-d, emeraldrapids, clearwaterforest, knl, knm, k8, athlon64, athlon-fx, opteron, k8-sse3, athlon64-sse3, opteron-sse3, amdfam10, barcelona, btver1, btver2, bdver1, bdver2, bdver3, bdver4, znver1, znver2, znver3, znver4, x86-64, x86-64-v2, x86-64-v3, x86-64-v4


In [28]:
def stream_gpt(python):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(python), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield reply.replace('```cpp\n','').replace('```','')

In [29]:
def stream_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            yield reply.replace('```cpp\n','').replace('```','')

In [30]:
def stream_gemini(python):
    # Create the prompt by combining system message and user prompt
    full_prompt = system_message + "\n\n" + user_prompt_for(python)
    
    try:
        response = gemini.generate_content(
            full_prompt,
            stream=True,
            generation_config=google.generativeai.types.GenerationConfig(
                max_output_tokens=2000,
                temperature=0.1
            )
        )
        
        result = ""
        for chunk in response:
            if chunk.text:
                result += chunk.text
                yield result.replace('```cpp\n','').replace('```','')
                
    except Exception as e:
        yield f"Error with Gemini: {str(e)}"

In [31]:
def optimize(python, model):
    if model=="GPT":
        result = stream_gpt(python)
    elif model=="Claude":
        result = stream_claude(python)
    elif model=="Gemini":
        result = stream_gemini(python)
    else:
        raise ValueError("Unknown model")
    for stream_so_far in result:
        yield stream_so_far

In [32]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=10, value=python_hard)
        cpp = gr.Textbox(label="C++ code:", lines=10)
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude", "Gemini"], label="Select model", value="GPT")
        convert = gr.Button("Convert code")

    convert.click(optimize, inputs=[python, model], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [33]:
def execute_python(code):
    try:
        output = io.StringIO()
        sys.stdout = output
        exec(code)
    finally:
        sys.stdout = sys.__stdout__
    return output.getvalue()

In [34]:
def execute_cpp(code):
    write_output(code)
    
    # Use the dynamic compiler detection from CloudLlama's contribution
    compiler_info = c_compiler_cmd("optimized")
    
    if compiler_info[1] == "Unavailable":
        return "❌ No C++ compiler available on this system"
    
    try:
        # Use the dynamically detected compiler command
        compile_cmd = compiler_info[2]  # This contains the full compiler command
        compile_result = subprocess.run(compile_cmd, check=True, text=True, capture_output=True)
        
        # Run the executable
        if compiler_info[0] == "Windows":
            run_cmd = ["./optimized.exe"]
        else:
            run_cmd = ["./optimized"]
            
        run_result = subprocess.run(run_cmd, check=True, text=True, capture_output=True)
        return run_result.stdout
        
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [35]:
# Docstring and comment generation system message
docstring_system_message = "You are an expert Python developer who adds comprehensive docstrings and comments to Python code. "
docstring_system_message += "Add Google-style docstrings to all functions and classes, and add inline comments to explain complex logic. "
docstring_system_message += "Preserve the original functionality exactly - only add documentation, don't change the code behavior. "
docstring_system_message += "Return only the documented Python code without any explanations or markdown formatting."

# 🆕 Docstring & Comments Generator

This notebook now includes a **comprehensive docstring and comments generation tool** that automatically enhances your Python code with professional documentation.

## ✨ Features:
- **Google-style docstrings** for functions and classes
- **Inline comments** explaining complex algorithms and logic
- **Multiple model support** (GPT, Claude, Gemini, CodeQwen, CodeQwen2.5)
- **Streaming output** for real-time generation
- **Code preservation** - only adds documentation, never changes functionality

## 🎯 Use Cases:
- **Code reviews** - Make your code more readable
- **Documentation** - Prepare code for sharing or production
- **Learning** - Understand complex algorithms through AI explanations
- **Professional development** - Follow Python documentation best practices

## 🚀 How to Use:
1. Paste your Python code into the input field
2. Select your preferred AI model
3. Click "Add Docstrings & Comments"
4. Get professionally documented code with comprehensive explanations

In [36]:
def docstring_prompt_for(python_code):
    """Create prompt for adding docstrings and comments to Python code"""
    user_prompt = "Add comprehensive Google-style docstrings to all functions and classes in this Python code. "
    user_prompt += "Also add inline comments to explain complex logic, algorithms, and non-obvious operations. "
    user_prompt += "Preserve the exact functionality - only add documentation, don't modify the code behavior. "
    user_prompt += "Return only the documented Python code:\n\n"
    user_prompt += python_code
    return user_prompt

def docstring_messages_for(python_code):
    """Create message format for docstring generation"""
    return [
        {"role": "system", "content": docstring_system_message},
        {"role": "user", "content": docstring_prompt_for(python_code)}
    ]

In [37]:
def stream_docstring_gpt(python_code):
    """Generate docstrings using GPT with streaming"""
    stream = openai.chat.completions.create(
        model=OPENAI_MODEL, 
        messages=docstring_messages_for(python_code), 
        stream=True
    )
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield reply.replace('```python\n','').replace('```','')

def stream_docstring_claude(python_code):
    """Generate docstrings using Claude with streaming"""
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=3000,
        system=docstring_system_message,
        messages=[{"role": "user", "content": docstring_prompt_for(python_code)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            yield reply.replace('```python\n','').replace('```','')

def stream_docstring_gemini(python_code):
    """Generate docstrings using Gemini with streaming"""
    full_prompt = docstring_system_message + "\n\n" + docstring_prompt_for(python_code)
    
    try:
        response = gemini.generate_content(
            full_prompt,
            stream=True,
            generation_config=google.generativeai.types.GenerationConfig(
                max_output_tokens=3000,
                temperature=0.1
            )
        )
        
        result = ""
        for chunk in response:
            if chunk.text:
                result += chunk.text
                yield result.replace('```python\n','').replace('```','')
                
    except Exception as e:
        yield f"Error with Gemini: {str(e)}"

In [ ]:
def stream_docstring_qwen(python_code):
    """Generate docstrings using CodeQwen with streaming"""
    tokenizer = AutoTokenizer.from_pretrained(code_qwen)
    messages = docstring_messages_for(python_code)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    client = InferenceClient(CODE_QWEN_URL, token=hf_token)
    stream = client.text_generation(text, stream=True, details=True, max_new_tokens=3000)
    result = ""
    for r in stream:
        result += r.token.text
        yield result.replace('```python\n','').replace('```','')

def stream_docstring_qwen25(python_code):
    """Generate docstrings using Qwen2.5-Coder with streaming"""
    client = InferenceClient(
        provider="together",
        api_key=os.environ["HF_TOKEN"],
    )
    
    messages = docstring_messages_for(python_code)
    
    try:
        completion = client.chat.completions.create(
            model=code_qwen25,
            messages=messages,
            stream=True,
            max_tokens=3000,
            temperature=0.1
        )
        
        result = ""
        for chunk in completion:
            if hasattr(chunk, 'choices') and chunk.choices:
                if hasattr(chunk.choices[0], 'delta') and chunk.choices[0].delta:
                    if hasattr(chunk.choices[0].delta, 'content') and chunk.choices[0].delta.content:
                        result += chunk.choices[0].delta.content
                        yield result.replace('```python\n','').replace('```','')
    except Exception as e:
        yield f"Error with Qwen2.5-Coder: {str(e)}"

def stream_docstring_codellama(python_code):
    """Generate docstrings using CodeLlama with streaming"""
    client = InferenceClient(CODELLAMA_URL, token=hf_token)
    messages = docstring_messages_for(python_code)
    
    try:
        completion = client.chat.completions.create(
            model="tgi",
            messages=messages,
            stream=True,
            max_tokens=3000,
            temperature=0.1
        )
        
        result = ""
        for chunk in completion:
            if hasattr(chunk, 'choices') and chunk.choices and len(chunk.choices) > 0:
                choice = chunk.choices[0]
                if hasattr(choice, 'delta') and choice.delta:
                    if hasattr(choice.delta, 'content') and choice.delta.content:
                        result += choice.delta.content
                        yield result.replace('```python\n','').replace('```','')
    except Exception as e:
        yield f"Error with CodeLlama: {str(e)}"

def stream_docstring_starcoder(python_code):
    """Generate docstrings using StarCoder with streaming"""
    client = InferenceClient(STARCODER_URL, token=hf_token)
    prompt = f"{docstring_system_message}\n\n{docstring_prompt_for(python_code)}"
    
    try:
        stream = client.text_generation(
            prompt, 
            stream=True, 
            details=True, 
            max_new_tokens=3000,
            temperature=0.1,
            do_sample=True
        )
        
        result = ""
        for r in stream:
            if hasattr(r, 'token') and r.token:
                result += r.token.text
                yield result.replace('```python\n','').replace('```','')
    except Exception as e:
        yield f"Error with StarCoder: {str(e)}"

def add_docstrings(python_code, model):
    """Main function to add docstrings using selected model"""
    if model == "GPT":
        result = stream_docstring_gpt(python_code)
    elif model == "Claude":
        result = stream_docstring_claude(python_code)
    elif model == "Gemini":
        result = stream_docstring_gemini(python_code)
    elif model == "CodeQwen":
        result = stream_docstring_qwen(python_code)
    elif model == "CodeQwen2.5":
        result = stream_docstring_qwen25(python_code)
    elif model == "CodeLlama":
        result = stream_docstring_codellama(python_code)
    elif model == "StarCoder":
        result = stream_docstring_starcoder(python_code)
    else:
        raise ValueError("Unknown model")
    
    for stream_so_far in result:
        yield stream_so_far

In [39]:
# Sample Python code without docstrings for testing
sample_undocumented_code = """
import math
import random

class DataProcessor:
    def __init__(self, data):
        self.data = data
        self.processed_data = []
    
    def filter_data(self, threshold):
        filtered = []
        for item in self.data:
            if item > threshold:
                filtered.append(item)
        return filtered
    
    def calculate_statistics(self):
        if not self.data:
            return None
        mean = sum(self.data) / len(self.data)
        variance = sum((x - mean) ** 2 for x in self.data) / len(self.data)
        std_dev = math.sqrt(variance)
        return {'mean': mean, 'variance': variance, 'std_dev': std_dev}

def generate_random_data(size, min_val=0, max_val=100):
    return [random.randint(min_val, max_val) for _ in range(size)]

def fibonacci(n):
    if n <= 1:
        return n
    a, b = 0, 1
    for i in range(2, n + 1):
        a, b = b, a + b
    return b

def binary_search(arr, target):
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1
"""

In [ ]:
# Docstring Generation Interface
css = """
.python {background-color: #306998;}
.cpp {background-color: #050;}
"""

with gr.Blocks(css=css) as docstring_ui:
    gr.Markdown("## 📝 Python Docstring & Comments Generator")
    gr.Markdown("This tool automatically adds Google-style docstrings and inline comments to your Python code.")
    
    with gr.Row():
        input_code = gr.Textbox(
            label="Python Code (without docstrings):", 
            value=sample_undocumented_code, 
            lines=15,
            placeholder="Paste your Python code here..."
        )
        documented_code = gr.Textbox(
            label="Python Code (with docstrings):", 
            lines=15
        )
    
    with gr.Row():
        doc_model = gr.Dropdown(
            ["GPT", "Claude", "Gemini", "CodeQwen", "CodeQwen2.5", "CodeLlama", "StarCoder"], 
            label="Select Model", 
            value="GPT"
        )
        
    with gr.Row():
        add_docs_btn = gr.Button("🚀 Add Docstrings & Comments", variant="primary")
        
    with gr.Row():
        gr.Markdown("""
        ### ℹ️ Features:
        - **Google-style docstrings** for all functions and classes
        - **Inline comments** explaining complex logic
        - **Type hints preservation** (if present)
        - **Original functionality maintained** - only documentation is added
        """)

    add_docs_btn.click(
        add_docstrings, 
        inputs=[input_code, doc_model], 
        outputs=[documented_code]
    )

# Launch the docstring interface
docstring_ui.launch(inbrowser=True, server_name="0.0.0.0", server_port=7861, show_api=False)

* Running on local URL:  http://0.0.0.0:7861
* To create a public link, set `share=True` in `launch()`.


In [41]:
# Test the docstring generation with GPT
print("🔥 Testing docstring generation with GPT...")
print("=" * 50)

test_code = """
def quick_sort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quick_sort(left) + middle + quick_sort(right)

def calculate_prime_numbers(limit):
    primes = []
    for num in range(2, limit + 1):
        is_prime = True
        for i in range(2, int(num ** 0.5) + 1):
            if num % i == 0:
                is_prime = False
                break
        if is_prime:
            primes.append(num)
    return primes
"""

# Generate docstrings
for result in stream_docstring_gpt(test_code):
    # Clear the output and show the current result
    print("\033[2J\033[H", end="")  # Clear screen and move cursor to top
    print("🔥 Testing docstring generation with GPT...")
    print("=" * 50)
    print(result)

🔥 Testing docstring generation with GPT...
🔥 Testing docstring generation with GPT...

🔥 Testing docstring generation with GPT...

🔥 Testing docstring generation with GPT...
python
🔥 Testing docstring generation with GPT...

🔥 Testing docstring generation with GPT...
def
🔥 Testing docstring generation with GPT...
def quick
🔥 Testing docstring generation with GPT...
def quick_sort
🔥 Testing docstring generation with GPT...
def quick_sort(arr
🔥 Testing docstring generation with GPT...
def quick_sort(arr):

🔥 Testing docstring generation with GPT...
def quick_sort(arr):
   
🔥 Testing docstring generation with GPT...
def quick_sort(arr):
    """
🔥 Testing docstring generation with GPT...
def quick_sort(arr):
    """Sort
🔥 Testing docstring generation with GPT...
def quick_sort(arr):
    """Sorts
🔥 Testing docstring generation with GPT...
def quick_sort(arr):
    """Sorts an
🔥 Testing docstring generation with GPT...
def quick_sort(arr):
    """Sorts an array
🔥 Testing docstring generation 

In [42]:
css = """
.python {background-color: #306998;}
.cpp {background-color: #050;}
"""

In [43]:
with gr.Blocks(css=css) as ui:
    gr.Markdown("## Convert code from Python to C++")
    with gr.Row():
        python = gr.Textbox(label="Python code:", value=python_hard, lines=10)
        cpp = gr.Textbox(label="C++ code:", lines=10)
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude", "Gemini"], label="Select model", value="GPT")
    with gr.Row():
        convert = gr.Button("Convert code")
    with gr.Row():
        python_run = gr.Button("Run Python")
        cpp_run = gr.Button("Run C++")
    with gr.Row():
        python_out = gr.TextArea(label="Python result:", elem_classes=["python"])
        cpp_out = gr.TextArea(label="C++ result:", elem_classes=["cpp"])

    convert.click(optimize, inputs=[python, model], outputs=[cpp])
    python_run.click(execute_python, inputs=[python], outputs=[python_out])
    cpp_run.click(execute_cpp, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [44]:
from huggingface_hub import login, InferenceClient
from transformers import AutoTokenizer

/home/hafnium/anaconda3/envs/llms/lib/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


In [45]:
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
code_qwen = "Qwen/CodeQwen1.5-7B-Chat"
code_qwen25 = "Qwen/Qwen2.5-Coder-32B-Instruct"
CODE_QWEN_URL = "https://vkvwxfaflgst818r.us-east4.gcp.endpoints.huggingface.cloud"
# Using Inference Providers for Qwen2.5-Coder-32B-Instruct (no URL needed)

# CodeLlama and StarCoder Endpoints
CODELLAMA_URL = "https://m48cv8d73jkzbe6v.us-east4.gcp.endpoints.huggingface.cloud"
STARCODER_URL = "https://nsqzesy2izi2auxq.us-east4.gcp.endpoints.huggingface.cloud"

In [47]:
tokenizer = AutoTokenizer.from_pretrained(code_qwen)
messages = messages_for(pi)
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [48]:
print(text)

<|im_start|>system
You are an assistant that reimplements Python code in high performance C++ for an M1 Mac. Respond only with C++ code; use comments sparingly and do not provide any explanation other than occasional comments. The C++ response needs to produce an identical output in the fastest possible time. Keep implementations of random number generators identical so that results match exactly.<|im_end|>
<|im_start|>user
Rewrite this Python code in C++ with the fastest possible implementation that produces identical output in the least time. Respond only with C++ code; do not explain your work other than a few comments. Pay attention to number types to ensure no int overflows. Remember to #include all necessary C++ packages such as iomanip.


import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

sta

In [49]:
client = InferenceClient(CODE_QWEN_URL, token=hf_token)
stream = client.text_generation(text, stream=True, details=True, max_new_tokens=3000)
for r in stream:
    print(r.token.text, end = "")

Here is the C++ code that achieves the same result as the Python code:

```cpp
#include <iostream>
#include <iomanip>
#include <chrono>

double calculate(int iterations, double param1, double param2) {
    double result = 1.0;
    for (int i = 1; i <= iterations; ++i) {
        double j = i * param1 - param2;
        result -= 1.0 / j;
        j = i * param1 + param2;
        result += 1.0 / j;
    }
    return result;
}

int main() {
    auto start_time = std::chrono::high_resolution_clock::now();
    double result = calculate(100000000, 4, 1) * 4;
    auto end_time = std::chrono::high_resolution_clock::now();

    std::cout << "Result: " << std::setprecision(12) << result << std::endl;
    std::cout << "Execution Time: " << std::chrono::duration<double>(end_time - start_time).count() << " seconds" << std::endl;

    return 0;
}
```

This C++ code does the same thing as the Python code: it calculates the result of a mathematical expression for a given number of iterations, and then pr

In [50]:
def stream_code_qwen(python):
    tokenizer = AutoTokenizer.from_pretrained(code_qwen)
    messages = messages_for(python)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    client = InferenceClient(CODE_QWEN_URL, token=hf_token)
    stream = client.text_generation(text, stream=True, details=True, max_new_tokens=3000)
    result = ""
    for r in stream:
        result += r.token.text
        yield result    

In [51]:
def stream_code_qwen25(python):
    """Stream function for Qwen2.5-Coder-32B-Instruct using Inference Providers"""
    # Create client using Inference Providers with Together
    client = InferenceClient(
        provider="together",
        api_key=os.environ["HF_TOKEN"],
    )
    
    # Prepare messages for chat completion
    messages = messages_for(python)
    
    try:
        # Use chat completions API for streaming
        completion = client.chat.completions.create(
            model=code_qwen25,
            messages=messages,
            stream=True,
            max_tokens=3000,
            temperature=0.1
        )
        
        result = ""
        for chunk in completion:
            if hasattr(chunk, 'choices') and chunk.choices:
                if hasattr(chunk.choices[0], 'delta') and chunk.choices[0].delta:
                    if hasattr(chunk.choices[0].delta, 'content') and chunk.choices[0].delta.content:
                        result += chunk.choices[0].delta.content
                        yield result
    except Exception as e:
        yield f"Error with Qwen2.5-Coder: {str(e)}"

In [ ]:
def stream_codellama(python):
    """Stream function for CodeLlama-7B-Instruct using HuggingFace Endpoints"""
    client = InferenceClient(CODELLAMA_URL, token=hf_token)
    
    # CodeLlama uses OpenAI-compatible API format
    messages = messages_for(python)
    
    try:
        # Use chat completions API for streaming
        completion = client.chat.completions.create(
            model="tgi",
            messages=messages,
            stream=True,
            max_tokens=3000,
            temperature=0.1
        )
        
        result = ""
        for chunk in completion:
            if hasattr(chunk, 'choices') and chunk.choices and len(chunk.choices) > 0:
                choice = chunk.choices[0]
                if hasattr(choice, 'delta') and choice.delta:
                    if hasattr(choice.delta, 'content') and choice.delta.content:
                        result += choice.delta.content
                        yield result
    except Exception as e:
        yield f"Error with CodeLlama: {str(e)}"

def stream_starcoder(python):
    """Stream function for StarCoder2-3B using HuggingFace Endpoints"""
    client = InferenceClient(STARCODER_URL, token=hf_token)
    
    # Create a prompt for StarCoder (it's not a chat model)
    prompt = f"{system_message}\n\n{user_prompt_for(python)}"
    
    try:
        # Use text generation API for streaming
        stream = client.text_generation(
            prompt, 
            stream=True, 
            details=True, 
            max_new_tokens=3000,
            temperature=0.1,
            do_sample=True
        )
        
        result = ""
        for r in stream:
            if hasattr(r, 'token') and r.token:
                result += r.token.text
                yield result
    except Exception as e:
        yield f"Error with StarCoder: {str(e)}"

In [52]:
# Test Qwen2.5-Coder-32B-Instruct with Inference Providers
def test_qwen25():
    """Test function to verify Qwen2.5-Coder is working"""
    try:
        client = InferenceClient(
            provider="together",
            api_key=os.environ["HF_TOKEN"],
        )
        
        completion = client.chat.completions.create(
            model=code_qwen25,
            messages=[
                {
                    "role": "user",
                    "content": "What is the capital of France?"
                }
            ],
            max_tokens=100
        )
        
        print("✅ Qwen2.5-Coder test successful!")
        print("Response:", completion.choices[0].message.content)
        return True
        
    except Exception as e:
        print(f"❌ Qwen2.5-Coder test failed: {str(e)}")
        return False

# Uncomment the line below to test the connection
# test_qwen25()

In [ ]:
# Test CodeLlama and StarCoder endpoints
def test_codellama():
    """Test function to verify CodeLlama is working"""
    try:
        client = InferenceClient(CODELLAMA_URL, token=hf_token)
        
        completion = client.chat.completions.create(
            model="tgi",
            messages=[
                {
                    "role": "user", 
                    "content": "Write a simple Python function to calculate factorial"
                }
            ],
            max_tokens=200
        )
        
        print("✅ CodeLlama test successful!")
        print("Response:", completion.choices[0].message.content[:200] + "...")
        return True
        
    except Exception as e:
        print(f"❌ CodeLlama test failed: {str(e)}")
        return False

def test_starcoder():
    """Test function to verify StarCoder is working"""
    try:
        client = InferenceClient(STARCODER_URL, token=hf_token)
        
        response = client.text_generation(
            "def fibonacci(n):", 
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True
        )
        
        print("✅ StarCoder test successful!")
        print("Response:", response[:200] + "...")
        return True
        
    except Exception as e:
        print(f"❌ StarCoder test failed: {str(e)}")
        return False

# Uncomment the lines below to test the connections
# test_codellama()
# test_starcoder()

# 🆕 Qwen2.5-Coder-32B-Instruct Integration

This notebook now includes **Qwen2.5-Coder-32B-Instruct** using **HuggingFace Inference Providers** with Together AI.

## Model Comparison:
- **CodeQwen1.5-7B-Chat**: Uses HuggingFace Inference Endpoints (original implementation)
- **Qwen2.5-Coder-32B-Instruct**: Uses HuggingFace Inference Providers via Together AI (new implementation)

## Key Benefits of Qwen2.5-Coder:
- 🧠 **Larger Model**: 32B parameters vs 7B (better code quality)
- 💰 **Cost Efficient**: Pay-per-use with Inference Providers (no endpoint hosting costs)  
- 🚀 **Latest Architecture**: Qwen2.5 generation with improved coding capabilities
- 🔧 **Easy Setup**: No need to deploy custom endpoints

## Usage:
Select "CodeQwen2.5" from the model dropdown in the interface below to use the new model.

# 🆕 CodeLlama & StarCoder Integration

This notebook now includes **CodeLlama-7B-Instruct** and **StarCoder2-3B** using **HuggingFace Inference Endpoints**.

## 🦙 CodeLlama-7B-Instruct Features:
- **Meta's specialized coding model** fine-tuned for instruction following
- **Excellent at code generation** and explanation tasks
- **OpenAI-compatible API** through HuggingFace Endpoints
- **7B parameters** providing good balance of speed and quality

## ⭐ StarCoder2-3B Features:  
- **BigCode's efficient coding model** optimized for code completion
- **Lightweight 3B parameters** for fast inference
- **Specialized for multiple programming languages**
- **Text generation API** for flexible code generation

## 🔗 Endpoint Integration:
Both models are deployed on **HuggingFace Inference Endpoints** with:
- **Scale-to-zero** billing (only pay when using)
- **TLS/SSL security** with token authentication  
- **Automatic scaling** based on demand
- **$0.70/hour** when running (paused when idle)

## 🎯 Usage:
Select "CodeLlama" or "StarCoder" from the model dropdown in any interface to use these models for C++ conversion or docstring generation.

In [ ]:
def optimize(python, model):
    if model=="GPT":
        result = stream_gpt(python)
    elif model=="Claude":
        result = stream_claude(python)
    elif model=="Gemini":
        result = stream_gemini(python)
    elif model=="CodeQwen":
        result = stream_code_qwen(python)
    elif model=="CodeQwen2.5":
        result = stream_code_qwen25(python)
    elif model=="CodeLlama":
        result = stream_codellama(python)
    elif model=="StarCoder":
        result = stream_starcoder(python)
    else:
        raise ValueError("Unknown model")
    for stream_so_far in result:
        yield stream_so_far

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Thank you to @CloudLlama for an amazing contribution</h2>
            <span style="color:#090;">
                A student has contributed a chunk of code to improve this, in the next 2 cells. You can now select which Python porgram to run,
                and a compiler is automatically selected that will work on PC, Windows and Mac. Massive thank you @CloudLlama!
            </span>
        </td>
    </tr>
</table>

In [54]:
def select_sample_program(sample_program):
    if sample_program=="pi":
        return pi
    elif sample_program=="python_hard":
        return python_hard
    else:
        return "Type your Python program here"

In [55]:
import platform

VISUAL_STUDIO_2022_TOOLS = "C:\\Program Files\\Microsoft Visual Studio\\2022\\Community\\Common7\Tools\\VsDevCmd.bat"
VISUAL_STUDIO_2019_TOOLS = "C:\\Program Files (x86)\\Microsoft Visual Studio\\2019\\BuildTools\\Common7\\Tools\\VsDevCmd.bat"

simple_cpp = """
#include <iostream>

int main() {
    std::cout << "Hello";
    return 0;
}
"""

def run_cmd(command_to_run):
    try:
        run_result = subprocess.run(command_to_run, check=True, text=True, capture_output=True)
        return run_result.stdout if run_result.stdout else "SUCCESS"
    except:
        return ""

def c_compiler_cmd(filename_base):
    my_platform = platform.system()
    my_compiler = []

    try:
        with open("simple.cpp", "w") as f:
            f.write(simple_cpp)
            
        if my_platform == "Windows":
            if os.path.isfile(VISUAL_STUDIO_2022_TOOLS):
                if os.path.isfile("./simple.exe"):
                    os.remove("./simple.exe")
                compile_cmd = ["cmd", "/c", VISUAL_STUDIO_2022_TOOLS, "&", "cl", "simple.cpp"]
                if run_cmd(compile_cmd):
                    if run_cmd(["./simple.exe"]) == "Hello":
                        my_compiler = ["Windows", "Visual Studio 2022", ["cmd", "/c", VISUAL_STUDIO_2022_TOOLS, "&", "cl", f"{filename_base}.cpp"]]
        
            if not my_compiler:
                if os.path.isfile(VISUAL_STUDIO_2019_TOOLS):
                    if os.path.isfile("./simple.exe"):
                        os.remove("./simple.exe")
                    compile_cmd = ["cmd", "/c", VISUAL_STUDIO_2019_TOOLS, "&", "cl", "simple.cpp"]
                    if run_cmd(compile_cmd):
                        if run_cmd(["./simple.exe"]) == "Hello":
                            my_compiler = ["Windows", "Visual Studio 2019", ["cmd", "/c", VISUAL_STUDIO_2019_TOOLS, "&", "cl", f"{filename_base}.cpp"]]
    
            if not my_compiler:
                my_compiler=[my_platform, "Unavailable", []]
                
        elif my_platform == "Linux":
            if os.path.isfile("./simple"):
                os.remove("./simple")
            compile_cmd = ["g++", "simple.cpp", "-o", "simple"]
            if run_cmd(compile_cmd):
                if run_cmd(["./simple"]) == "Hello":
                    my_compiler = ["Linux", "GCC (g++)", ["g++", f"{filename_base}.cpp", "-o", f"{filename_base}" ]]
    
            if not my_compiler:
                if os.path.isfile("./simple"):
                    os.remove("./simple")
                compile_cmd = ["clang++", "simple.cpp", "-o", "simple"]
                if run_cmd(compile_cmd):
                    if run_cmd(["./simple"]) == "Hello":
                        my_compiler = ["Linux", "Clang++", ["clang++", f"{filename_base}.cpp", "-o", f"{filename_base}"]]
        
            if not my_compiler:
                my_compiler=[my_platform, "Unavailable", []]
    
        elif my_platform == "Darwin":
            if os.path.isfile("./simple"):
                os.remove("./simple")
            compile_cmd = ["clang++", "-Ofast", "-std=c++17", "-march=armv8.5-a", "-mtune=apple-m1", "-mcpu=apple-m1", "-o", "simple", "simple.cpp"]
            if run_cmd(compile_cmd):
                if run_cmd(["./simple"]) == "Hello":
                    my_compiler = ["Macintosh", "Clang++", ["clang++", "-Ofast", "-std=c++17", "-march=armv8.5-a", "-mtune=apple-m1", "-mcpu=apple-m1", "-o", f"{filename_base}", f"{filename_base}.cpp"]]
    
            if not my_compiler:
                my_compiler=[my_platform, "Unavailable", []]
    except:
        my_compiler=[my_platform, "Unavailable", []]
        
    if my_compiler:
        return my_compiler
    else:
        return ["Unknown", "Unavailable", []]


In [ ]:
compiler_cmd = c_compiler_cmd("optimized")

with gr.Blocks(css=css) as ui:
    gr.Markdown("## Convert code from Python to C++")
    with gr.Row():
        python = gr.Textbox(label="Python code:", value=python_hard, lines=10)
        cpp = gr.Textbox(label="C++ code:", lines=10)
    with gr.Row():
        with gr.Column():
            sample_program = gr.Radio(["pi", "python_hard"], label="Sample program", value="python_hard")
            model = gr.Dropdown(["GPT", "Claude", "Gemini", "CodeQwen", "CodeQwen2.5", "CodeLlama", "StarCoder"], label="Select model", value="GPT")
        with gr.Column():
            architecture = gr.Radio([compiler_cmd[0]], label="Architecture", interactive=False, value=compiler_cmd[0])
            compiler = gr.Radio([compiler_cmd[1]], label="Compiler", interactive=False, value=compiler_cmd[1])
    with gr.Row():
        convert = gr.Button("Convert code")
    with gr.Row():
        python_run = gr.Button("Run Python")
        if not compiler_cmd[1] == "Unavailable":
            cpp_run = gr.Button("Run C++")
        else:
            cpp_run = gr.Button("No compiler to run C++", interactive=False)
    with gr.Row():
        python_out = gr.TextArea(label="Python result:", elem_classes=["python"])
        cpp_out = gr.TextArea(label="C++ result:", elem_classes=["cpp"])

    sample_program.change(select_sample_program, inputs=[sample_program], outputs=[python])
    convert.click(optimize, inputs=[python, model], outputs=[cpp])
    python_run.click(execute_python, inputs=[python], outputs=[python_out])
    cpp_run.click(execute_cpp, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)
# ui.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Enhanced comprehensive interface with tabbed layout
with gr.Blocks(css=css, title="LLM Code Tools") as enhanced_ui:
    gr.Markdown("# 🚀 LLM Code Engineering Toolkit")
    gr.Markdown("Professional code conversion, optimization, and documentation tools powered by multiple AI models.")
    
    with gr.Tabs():
        # Tab 1: Python to C++ Conversion
        with gr.TabItem("🔄 Python → C++"):
            gr.Markdown("## Convert code from Python to C++")
            with gr.Row():
                python = gr.Textbox(label="Python code:", value=python_hard, lines=12)
                cpp = gr.Textbox(label="C++ code:", lines=12)
            with gr.Row():
                with gr.Column():
                    sample_program = gr.Radio(["pi", "python_hard"], label="Sample program", value="python_hard")
                    model = gr.Dropdown(["GPT", "Claude", "Gemini", "CodeQwen", "CodeQwen2.5", "CodeLlama", "StarCoder"], label="Select model", value="GPT")
                with gr.Column():
                    architecture = gr.Radio([compiler_cmd[0]], label="Architecture", interactive=False, value=compiler_cmd[0])
                    compiler = gr.Radio([compiler_cmd[1]], label="Compiler", interactive=False, value=compiler_cmd[1])
            with gr.Row():
                convert = gr.Button("🚀 Convert to C++", variant="primary")
            with gr.Row():
                python_run = gr.Button("▶️ Run Python")
                if not compiler_cmd[1] == "Unavailable":
                    cpp_run = gr.Button("⚡ Run C++")
                else:
                    cpp_run = gr.Button("❌ No compiler to run C++", interactive=False)
            with gr.Row():
                python_out = gr.TextArea(label="Python result:", elem_classes=["python"])
                cpp_out = gr.TextArea(label="C++ result:", elem_classes=["cpp"])

            # Event handlers for C++ conversion tab
            sample_program.change(select_sample_program, inputs=[sample_program], outputs=[python])
            convert.click(optimize, inputs=[python, model], outputs=[cpp])
            python_run.click(execute_python, inputs=[python], outputs=[python_out])
            cpp_run.click(execute_cpp, inputs=[cpp], outputs=[cpp_out])
        
        # Tab 2: Docstring Generation
        with gr.TabItem("📝 Add Docstrings"):
            gr.Markdown("## Python Docstring & Comments Generator")
            gr.Markdown("Automatically add Google-style docstrings and inline comments to your Python code.")
            
            with gr.Row():
                input_code = gr.Textbox(
                    label="Python Code (without docstrings):", 
                    value=sample_undocumented_code, 
                    lines=15,
                    placeholder="Paste your Python code here..."
                )
                documented_code = gr.Textbox(
                    label="Python Code (with docstrings):", 
                    lines=15
                )
            
            with gr.Row():
                doc_model = gr.Dropdown(
                    ["GPT", "Claude", "Gemini", "CodeQwen", "CodeQwen2.5", "CodeLlama", "StarCoder"], 
                    label="Select Model", 
                    value="GPT"
                )
                
            with gr.Row():
                add_docs_btn = gr.Button("📝 Add Docstrings & Comments", variant="primary")
                
            with gr.Row():
                gr.Markdown("""
                ### ✨ Features:
                - **Google-style docstrings** for all functions and classes  
                - **Inline comments** explaining complex logic  
                - **Type hints preservation** (if present)  
                - **Original functionality maintained** - only documentation is added
                """)

            # Event handlers for docstring tab
            add_docs_btn.click(
                add_docstrings, 
                inputs=[input_code, doc_model], 
                outputs=[documented_code]
            )

enhanced_ui.launch(inbrowser=True, server_name="0.0.0.0", server_port=7865, show_api=False)

* Running on local URL:  http://0.0.0.0:7865
* To create a public link, set `share=True` in `launch()`.


# 🎉 Enhanced Code Toolkit Summary

This notebook now includes **THREE powerful interfaces** for your code engineering needs:

## 🔧 Available Tools:

### 1. **Standalone Docstring Generator** (Port 7861)
   - Dedicated interface for adding docstrings and comments
   - Clean, focused UI for documentation tasks

### 2. **Original C++ Converter** (Original interface) 
   - The existing Python to C++ conversion tool
   - All **7 AI models** supported (GPT, Claude, Gemini, CodeQwen, CodeQwen2.5, CodeLlama, StarCoder)

### 3. **Enhanced Comprehensive Toolkit** (Port 7865) 
   - **Tabbed interface** combining both tools
   - **Tab 1**: Python → C++ conversion with compiler support
   - **Tab 2**: Docstring & comments generation
   - Professional UI with enhanced features

## 🤖 Model Lineup:

### **Closed Source Models:**
- **GPT-4o**: OpenAI's flagship model for code generation
- **Claude 3.5 Sonnet**: Anthropic's advanced reasoning model  
- **Google Gemini**: Fast and efficient code generation

### **Open Source Models:**
- **CodeQwen 1.5-7B**: Specialized coding model via HuggingFace Endpoints
- **Qwen2.5-Coder-32B**: Latest large coding model via Inference Providers
- **CodeLlama-7B-Instruct**: Meta's instruction-tuned coding model
- **StarCoder2-3B**: Efficient code generation model

## 🚀 Quick Start:
Run any of the interface cells above to launch the tool that best fits your workflow!

## 💡 Pro Tip:
Use the **Enhanced Comprehensive Toolkit** for the best experience - it includes both tools in a single, professional interface with tabbed navigation.